# 05 – GroupBy

So far you've been working with the entire DataFrame at once.

But a lot of real questions are about **groups** — not the whole data.

- What is the average marks per department?
- Which city has the highest total fees collected?
- How many students are in each grade?

To answer questions like these, you need `groupby()`.

`groupby()` splits your data into groups based on a column, applies a calculation to each group, and combines the results back into a table.

It follows a simple pattern called **Split → Apply → Combine**:

```
Split    →  group rows by Department
Apply    →  calculate mean Marks for each group
Combine  →  put the results into a single table
```

Once this clicks, you'll use it all the time.

## Setting Up

In [ ]:
import pandas as pd

data = {
    "Name":       ["Amit", "Priya", "Ravi", "Neha", "Karan", "Sneha", "Raj", "Anita"],
    "Department": ["Science", "Arts", "Science", "Commerce", "Arts", "Commerce", "Science", "Arts"],
    "City":       ["Pune", "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi", "Pune", "Delhi"],
    "Marks":      [85, 90, 78, 92, 88, 74, 81, 95],
    "Fees_Paid":  [12000, 15000, 12000, 18000, 15000, 18000, 12000, 15000]
}

df = pd.DataFrame(data)
print(df)

## 1) Basic GroupBy

Let's start simple — group by `Department` and find the average marks in each department.

In [ ]:
# Average marks per department

print(df.groupby("Department")["Marks"].mean())

Read this left to right:
- `df.groupby("Department")` — split the data into 3 groups: Arts, Commerce, Science
- `["Marks"]` — within each group, look at the Marks column
- `.mean()` — calculate the average

The result shows one row per group — exactly what you need.

In [ ]:
# You can swap mean() for any aggregation

print("Max marks per department:")
print(df.groupby("Department")["Marks"].max())
print()

print("Total fees collected per department:")
print(df.groupby("Department")["Fees_Paid"].sum())
print()

print("Number of students per department:")
print(df.groupby("Department")["Name"].count())

## 2) GroupBy on Multiple Columns at Once

Instead of calculating one column at a time, you can apply an aggregation to all numeric columns in one shot.

In [ ]:
# Mean of all numeric columns grouped by Department

print(df.groupby("Department").mean(numeric_only=True))

`numeric_only=True` tells pandas to skip text columns like Name and City — since you can't take an average of names.

## 3) Grouping by Multiple Columns

You can group by more than one column at the same time.

For example — average marks broken down by both Department AND City.

In [ ]:
# Group by Department and City together

print(df.groupby(["Department", "City"])["Marks"].mean())

The result has a two-level index — Department on the outside, City on the inside. Each row shows the average marks for that specific Department + City combination.

To get a flat table (easier to read and work with), add `reset_index()`.

In [ ]:
# Flat result with reset_index()

result = df.groupby(["Department", "City"])["Marks"].mean().reset_index()
print(result)

## 4) `agg()` — Multiple Aggregations at Once

What if you want the mean, max, and count all in one table?

`agg()` lets you apply multiple aggregation functions at the same time.

In [ ]:
# Multiple stats for Marks grouped by Department

result = df.groupby("Department")["Marks"].agg(["mean", "max", "min", "count"])
print(result)

In [ ]:
# Different aggregations for different columns
# agg() accepts a dictionary: {column: aggregation_function}

result = df.groupby("Department").agg({
    "Marks":     "mean",
    "Fees_Paid": "sum",
    "Name":      "count"
})

print(result)

This is one of the most useful patterns in pandas — one `groupby().agg()` call gives you a full summary table broken down by category.

Let's rename those columns to make the result cleaner.

In [ ]:
# Same result with clean column names using named aggregations

result = df.groupby("Department").agg(
    Avg_Marks   = ("Marks",     "mean"),
    Total_Fees  = ("Fees_Paid", "sum"),
    Num_Students= ("Name",      "count")
).reset_index()

print(result)

Named aggregations — `new_col_name = ("source_col", "function")` — let you name the output columns directly. Much cleaner than renaming afterwards.

## 5) `value_counts()` — Quick Frequency Count

When you just want to know how many times each value appears in a column — `value_counts()` is faster than a full `groupby`.

In [ ]:
# How many students are in each department?

print(df["Department"].value_counts())

In [ ]:
# As a percentage instead of count

print(df["Department"].value_counts(normalize=True) * 100)

`normalize=True` gives you proportions (0 to 1). Multiply by 100 to get percentages.

## 6) Filtering After GroupBy

A common pattern — group, aggregate, then filter the results to find only the groups that meet a condition.

In [ ]:
# Which departments have an average marks above 85?

dept_avg = df.groupby("Department")["Marks"].mean().reset_index()
dept_avg.columns = ["Department", "Avg_Marks"]

print(dept_avg[dept_avg["Avg_Marks"] > 85])

In [ ]:
# Which cities collected more than 35000 in total fees?

city_fees = df.groupby("City")["Fees_Paid"].sum().reset_index()
city_fees.columns = ["City", "Total_Fees"]

print(city_fees[city_fees["Total_Fees"] > 35000])

The pattern is:
1. `groupby()` + aggregation → summary table
2. `reset_index()` → flat table
3. filter on the result → only the groups that qualify

## 7) Sorting GroupBy Results

After grouping, you'll often want to sort the results — like ranking departments by average marks.

In [ ]:
# Departments ranked by average marks, highest first

result = (
    df.groupby("Department")["Marks"]
    .mean()
    .reset_index()
    .sort_values("Marks", ascending=False)
    .reset_index(drop=True)
)

print(result)

Chaining multiple steps with `()` around the whole expression keeps the code readable — each step on its own line, no intermediate variables needed.

## 8) Putting It All Together — Full Summary Report

Let's build a complete department summary from scratch.

In [ ]:
summary = df.groupby("Department").agg(
    Students     = ("Name",      "count"),
    Avg_Marks    = ("Marks",     "mean"),
    Top_Score    = ("Marks",     "max"),
    Total_Fees   = ("Fees_Paid", "sum")
).reset_index()

# Round the average marks to 1 decimal place
summary["Avg_Marks"] = summary["Avg_Marks"].round(1)

# Sort by Avg_Marks descending
summary = summary.sort_values("Avg_Marks", ascending=False).reset_index(drop=True)

print(summary)

In one block — a clean department report showing student count, average marks, top score, and total fees, sorted by performance. This is exactly the kind of output you'd put in a real report or dashboard.

## Summary

- `groupby()` follows **Split → Apply → Combine**
- Basic usage: `df.groupby("col")["other_col"].mean()`
- Common aggregations: `.mean()`, `.sum()`, `.max()`, `.min()`, `.count()`
- Group by multiple columns: `df.groupby(["col1", "col2"])`
- Multiple aggregations: `.agg(["mean", "max", "count"])`
- Different agg per column: `.agg({"col1": "mean", "col2": "sum"})`
- Named aggregations: `.agg(new_name=("col", "func"))` — cleanest approach
- Quick frequency count: `df["col"].value_counts()`
- Always use `reset_index()` after groupby to get a flat, usable table

Next up: **Merge and Join** — combining two DataFrames into one.

## Practice Questions 1

1. Create a DataFrame of 8 sales records with columns `Salesperson`, `Region`, `Product`, `Amount`.
2. Find the total sales amount per Region.
3. Find the average amount per Salesperson.
4. Count how many sales each Region has using `value_counts()`.

## Practice Questions 2

1. Use `agg()` to get the total, average, and max `Amount` per `Region` in one table.
2. Filter the result to show only regions with total sales above a threshold you choose.
3. Build a full summary table — `Region`, number of sales, average amount, top amount — sorted by average amount descending.